In [1]:
import numpy as np
import pandas as pd
import os
from scipy.stats import sem
import scipy

## Configuration

In [15]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Reverse_Probe" # "Fine_Tuned" | "Reverse_Probe"
model_name = "CLIP_ViT_Vision" # CLIP_ViT_Vision | DeiT 
domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
indices = [i for i in range(12)]
size = [i for i in range(1,6)]

## Loading Data

In [3]:
def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)
    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        indice = arr.index(max_val)
        num_img = data[indice]["Train_Data_Size"][i][0]
        best_acc[i] = (max_val, num_img)

    final = []
    num_img = []
    for i in indices:
        final.append(best_acc[i][0])
        num_img.append(best_acc[i][1])

    best_accuracy = max(final)
    index = final.index(best_accuracy)
    best_accuracy_num_images = num_img[index]

    return best_accuracy, best_accuracy_num_images, index

In [5]:
dataset_name = ["INaturalist", "Cifar10", "Cifar100", "Food101", "DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]

In [4]:
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

CLIP_ViT_Vision - DTD: Fine Tuned Accuracy
Fine-Tuned Average: 0.77489361702, Error: +- 0.014881606646707901, 95% Confidence Interval: (np.float64(0.7600120103732921), np.float64(0.7897752236667079))

CLIP_ViT_Vision - EuroSAT: Fine Tuned Accuracy
Fine-Tuned Average: 0.9830370370400001, Error: +- 0.005883891607124547, 95% Confidence Interval: (np.float64(0.9771531454328756), np.float64(0.9889209286471247))

CLIP_ViT_Vision - GTSRB: Fine Tuned Accuracy
Fine-Tuned Average: 0.9879334917000001, Error: +- 0.0011265049431746776, 95% Confidence Interval: (np.float64(0.9868069867568254), np.float64(0.9890599966431748))

CLIP_ViT_Vision - MNIST: Fine Tuned Accuracy
Fine-Tuned Average: 0.9945999999999999, Error: +- 0.0012570889871231783, 95% Confidence Interval: (np.float64(0.9933429110128768), np.float64(0.9958570889871231))

CLIP_ViT_Vision - RESISC45: Fine Tuned Accuracy
Fine-Tuned Average: 0.92390476192, Error: +- 0.00594152186901542, 95% Confidence Interval: (np.float64(0.9179632400509846),

In [5]:
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Linear_Probe/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Linear Probe Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

CLIP_ViT_Vision - DTD: Linear Probe Accuracy
Fine-Tuned Average: 0.7720212765800001, Error: +- 0.003605407605837163, 95% Confidence Interval: (np.float64(0.7684158689741629), np.float64(0.7756266841858372))

CLIP_ViT_Vision - EuroSAT: Linear Probe Accuracy
Fine-Tuned Average: 0.95955555554, Error: +- 0.0016388612732198826, 95% Confidence Interval: (np.float64(0.9579166942667802), np.float64(0.9611944168132199))

CLIP_ViT_Vision - GTSRB: Linear Probe Accuracy
Fine-Tuned Average: 0.8681868567000001, Error: +- 0.001767406353933354, 95% Confidence Interval: (np.float64(0.8664194503460667), np.float64(0.8699542630539334))

CLIP_ViT_Vision - MNIST: Linear Probe Accuracy
Fine-Tuned Average: 0.98758, Error: +- 0.0006044748860053462, 95% Confidence Interval: (np.float64(0.9869755251139947), np.float64(0.9881844748860054))

CLIP_ViT_Vision - RESISC45: Linear Probe Accuracy
Fine-Tuned Average: 0.9174603174600001, Error: +- 0.002033941000778694, 95% Confidence Interval: (np.float64(0.9154263764592

In [6]:
for k in range(0, len(dataset_name)):
    refined_acc = []
    refined_labels = []
    folder_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance/1/{domain}/{dataset_name[k]}_"
    
    for i in range(1,6):
        df = pd.read_json(f"{folder_path}/{i}.json")
        temp_acc = []
        for j in indices:
            temp_acc.append(df["Classification_Accuracy"][dataset_name[k]][str(j)])
        max_val = max(temp_acc)
        layer = temp_acc.index(max_val)
        refined_acc.append(max_val)
        refined_labels.append(layer)

    print(f"{model_name} - {dataset_name[k]}: Task Matrix Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Layers: {refined_labels}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

CLIP_ViT_Vision - DTD: Task Matrix Accuracy
Fine-Tuned Average: 0.7561702127659571, Layers: [11, 11, 11, 11, 11], Error: +- 0.007363457831534914, 95% Confidence Interval: (np.float64(0.7488067549344222), np.float64(0.763533670597492))

CLIP_ViT_Vision - EuroSAT: Task Matrix Accuracy
Fine-Tuned Average: 0.962, Layers: [6, 6, 6, 6, 8], Error: +- 0.0038005869842461237, 95% Confidence Interval: (np.float64(0.9581994130157538), np.float64(0.9658005869842461))

CLIP_ViT_Vision - GTSRB: Task Matrix Accuracy
Fine-Tuned Average: 0.865193982581156, Layers: [11, 11, 11, 11, 11], Error: +- 0.0013844016096218326, 95% Confidence Interval: (np.float64(0.8638095809715342), np.float64(0.8665783841907778))

CLIP_ViT_Vision - MNIST: Task Matrix Accuracy
Fine-Tuned Average: 0.99014, Layers: [7, 8, 7, 8, 8], Error: +- 0.0005013002154028445, 95% Confidence Interval: (np.float64(0.9896386997845972), np.float64(0.9906413002154029))

CLIP_ViT_Vision - RESISC45: Task Matrix Accuracy
Fine-Tuned Average: 0.891111

In [8]:
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Reverse_Probe/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Reverse Probe Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

CLIP_ViT_Vision - DTD: Reverse Probe Accuracy
Fine-Tuned Average: 0.7590425531999999, Error: +- 0.007856415761770541, 95% Confidence Interval: (np.float64(0.7511861374382294), np.float64(0.7668989689617705))

CLIP_ViT_Vision - EuroSAT: Reverse Probe Accuracy
Fine-Tuned Average: 0.98481481482, Error: +- 0.006624443099835009, 95% Confidence Interval: (np.float64(0.978190371720165), np.float64(0.991439257919835))

CLIP_ViT_Vision - GTSRB: Reverse Probe Accuracy
Fine-Tuned Average: 0.99015043546, Error: +- 0.0017163593396752885, 95% Confidence Interval: (np.float64(0.9884340761203247), np.float64(0.9918667947996753))

CLIP_ViT_Vision - MNIST: Reverse Probe Accuracy
Fine-Tuned Average: 0.9950000000000001, Error: +- 0.0006082886455166525, 95% Confidence Interval: (np.float64(0.9943917113544835), np.float64(0.9956082886455168))

CLIP_ViT_Vision - RESISC45: Reverse Probe Accuracy
Fine-Tuned Average: 0.94034920634, Error: +- 0.007274178938845521, 95% Confidence Interval: (np.float64(0.933075027

In [9]:
for k in range(0, len(dataset_name)):
    said = k
    results_path = f"./{model_name}/Data/Core/{refining_type}/{refiner}/{dataset_name[k]}/{domain}" # /Entire_Transformation_Matrix_W"

    results = {}

    for i in size:
        results[i] = []
        path = f"{results_path}/{i}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{i}.json", f"Base_Linear_Probe_Results_{i}.json", ".ipynb_checkpoints"]: 
                    continue
                file_path = os.path.join(path, filename)
                if os.path.isfile(file_path):
                    results[i].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = []
        for filepath in results[i]:
            try:
                df = pd.read_json(filepath)
                data.append(df)
            except ValueError as ve:
                print(f"Failed to read JSON from file: {filepath} | Error: {ve}")
            except Exception as e:
                print(f"Other error with file {filepath}: {e}")
        results[i] = data

        # data = [pd.read_json(i) for i in results[i]]
        # data = sorted(data, key=lambda df: df["Train_Data_Size"][0])
        results[i] = data
    
    best_acc = []
    num_img = []
    index = []
    
    print(f"{model_name} - {dataset_name[said]}: {domain}")
    for i in size:
        acc, img, ind = find_best_acc(results[i])
        best_acc.append(acc)
        num_img.append(img)
        index.append(ind)
        print(f"Set {i} Best Accuracy: {best_acc[i-1]} | Number of training images: {num_img[i-1]} | Transformation Layer (0-11): {index[i-1]}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(best_acc)-1, loc=np.mean(best_acc), scale=sem(best_acc))
    print(f"Average: {np.mean(best_acc)}, Error: +- {ci_high-np.mean(best_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

CLIP_ViT_Vision - DTD: Base_Fine_Tuned
Set 1 Best Accuracy: 0.7526595745 | Number of training images: 3760 | Transformation Layer (0-11): 11
Set 2 Best Accuracy: 0.7627659574000001 | Number of training images: 3008 | Transformation Layer (0-11): 11
Set 3 Best Accuracy: 0.7617021277 | Number of training images: 3008 | Transformation Layer (0-11): 11
Set 4 Best Accuracy: 0.7569148936000001 | Number of training images: 3760 | Transformation Layer (0-11): 11
Set 5 Best Accuracy: 0.7558510638 | Number of training images: 3760 | Transformation Layer (0-11): 11
Average: 0.7579787233999999, Error: +- 0.0052213913749619145, 95% Confidence Interval: (np.float64(0.752757332025038), np.float64(0.7632001147749619))

CLIP_ViT_Vision - EuroSAT: Base_Fine_Tuned
Set 1 Best Accuracy: 0.9666666667 | Number of training images: 21600 | Transformation Layer (0-11): 6
Set 2 Best Accuracy: 0.9611111111 | Number of training images: 21600 | Transformation Layer (0-11): 6
Set 3 Best Accuracy: 0.9651851852000001 

In [16]:
def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)

    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        best_acc[i] = max_val

    final = []
    for i in indices:
        final.append(best_acc[i])

    best_accuracy = max(final)
    index = final.index(best_accuracy)

    return best_accuracy, index

# Ablation Base with Fine Tuned Classifier Head
big_data = {i: {} for i in range(len(dataset_name))}

for k in range(len(dataset_name)):
    said=k
    results_path = f"./{model_name}/Data/Core/{refining_type}/{refiner}/{dataset_name[k]}/{domain}" # /Entire_Transformation_Matrix_W"

    for i in size:
        results[i] = []
        path = f"{results_path}/{i}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [f"Base_Fine_Tuned_Classifier_Results_{i}.json"]: 
                    file_path = os.path.join(path, filename)
                    if os.path.isfile(file_path):
                        results[i].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = [pd.read_json(i) for i in results[i]]
        results[i] = data
        big_data = results[i]
    
    best_acc = []
    index = []
    print(f"{model_name} - {dataset_name[said]}: Ablation Base with Fine-Tuned Classifier")
    for i in size:
        acc, ind = find_best_acc(results[i])
        best_acc.append(acc)
        index.append(ind)
        print(f"Set {i} Best Accuracy: {best_acc[i-1]} | Transformation Layer (0-11): {index[i-1]}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(best_acc)-1, loc=np.mean(best_acc), scale=sem(best_acc))
    print(f"Average: {np.mean(best_acc)}, Error: +- {ci_high-np.mean(best_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

CLIP_ViT_Vision - DTD: Ablation Base with Fine-Tuned Classifier
Set 1 Best Accuracy: 0.0340425532 | Transformation Layer (0-11): 8
Set 2 Best Accuracy: 0.0271276596 | Transformation Layer (0-11): 11
Set 3 Best Accuracy: 0.0436170213 | Transformation Layer (0-11): 11
Set 4 Best Accuracy: 0.0265957447 | Transformation Layer (0-11): 11
Set 5 Best Accuracy: 0.027659574500000002 | Transformation Layer (0-11): 9
Average: 0.03180851066, Error: +- 0.009012309088112756, 95% Confidence Interval: (np.float64(0.02279620157188724), np.float64(0.040820819748112754))

CLIP_ViT_Vision - EuroSAT: Ablation Base with Fine-Tuned Classifier
Set 1 Best Accuracy: 0.1566666667 | Transformation Layer (0-11): 5
Set 2 Best Accuracy: 0.0974074074 | Transformation Layer (0-11): 0
Set 3 Best Accuracy: 0.1588888889 | Transformation Layer (0-11): 2
Set 4 Best Accuracy: 0.2025925926 | Transformation Layer (0-11): 8
Set 5 Best Accuracy: 0.1277777778 | Transformation Layer (0-11): 0
Average: 0.14866666668, Error: +- 0.0